In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import QiskitRuntimeService


from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_aer import AerSimulator
service = QiskitRuntimeService()
from qiskit import transpile
import json
import math

In [ ]:
service = QiskitRuntimeService()

backend1 = service.backend('ibm_fez')
properties = backend1.properties()
sum = 0
for i in range(len(properties.qubits)):
    sum += properties.t1(i)

t1 = sum/backend1.num_qubits

print(t1)
print(f"Selected backend: {backend1.name}")
print(f"Number of qubits: {backend1.num_qubits}")
print(f"Pending jobs: {backend1.status().pending_jobs}")

In [ ]:
def gather_stats(backend):
    target = backend.target

    print(f"Backend name: {backend.name}")
    print(f"Number of qubits: {backend.num_qubits}")
    print(f"Backend version: {backend.backend_version}")
    print(f"Maximum circuits per job: {backend.max_circuits}")

    backend_properties = backend.properties(refresh=True)

    print(f"Backend: {backend.name}")
    print(f"Properties updated: {backend_properties.last_update_date}")

    qubit_0_properties = backend_properties.qubit_property(0)

    for property_name, property_data in qubit_0_properties.items():
        value, measured_at = property_data

        print(f"{property_name}: {value}")

    gate_records = []

    excluded_instructions = {
        "measure",
        "reset",
        "delay",
        "barrier"
    }

    for gate_name in sorted(target.operation_names):
        if gate_name in excluded_instructions:
            continue

        qubit_arguments = target.qargs_for_operation_name(gate_name)

        if qubit_arguments is None:
            continue

        for qubits_used in qubit_arguments:
            if len(qubits_used) not in (1, 2):
                continue

            gate_properties = target[gate_name][qubits_used]

            if gate_properties is None:
                continue

            gate_records.append(
                {
                    "gate": gate_name,
                    "qubits": qubits_used,
                    "num_qubits": len(qubits_used),
                    "error": gate_properties.error,
                    "duration": gate_properties.duration
                }
            )
    one_qubit_gates = sorted({
        record["gate"]
        for record in gate_records
        if record["num_qubits"] == 1
    })

    two_qubit_gates = sorted({
        record["gate"]
        for record in gate_records
        if record["num_qubits"] == 2
    })

    print("One-qubit gates:", one_qubit_gates)
    print("Two-qubit gates:", two_qubit_gates)

    print(
        f"{'Gate':>10} "
        f"{'Qubits':>8} "
        f"{'Instances':>10} "
        f"{'Median Error':>15} "
        f"{'Median Duration':>18}"
    )

    results = {}
    duration_results = {}
    gate_names = {record["gate"] for record in gate_records}

    for gate_name in gate_names:
        records = [
            record
            for record in gate_records
            if record["gate"] == gate_name
        ]

        errors = [
            record["error"]
            for record in records
            if record["error"] is not None
        ]

        durations = [
            record["duration"]
            for record in records
            if record["duration"] is not None
        ]

        gate_size = records[0]["num_qubits"]

        median_error = (
            100 * np.median(errors)
            if errors else np.nan
        )

        median_duration = (
            1e9 * np.median(durations)
            if durations else np.nan
        )

        results[gate_name] = {"avg_error_pct": median_error}
        duration_results[gate_name] = {"duration": median_duration}
        print(
            f"{gate_name:>10} "
            f"{gate_size:>8} "
            f"{len(records):>10} "
            f"{median_error:>14.4f}% "
            f"{median_duration:>15.2f} ns"
        )
    return results, duration_results

In [ ]:
import numpy as np
from qiskit import transpile
from qiskit.converters import circuit_to_dag


def accumulate_circuit_error(circuit, backend, gate_error_dict, gate_durration_dict, optimization_level=1,
                              name_aliases=None, tvalues=False):
    if name_aliases is None:
        name_aliases = {
            "measure_2": "measure",
            "reset_2": "reset",
        }

    dict_key_to_qiskit_name = {
        key: name_aliases.get(key, key) for key in gate_error_dict
    }
    
    basis_gates = sorted(set(dict_key_to_qiskit_name.values()))

    qiskit_name_to_dict_key = {v: k for k, v in dict_key_to_qiskit_name.items()}

    isa_circuit = transpile(
        circuit,
        backend=backend,
        basis_gates=basis_gates,
        optimization_level=optimization_level,
    )

    last_op = {}
    total_time_passed = 0

    def lookup_error(gate_name, dict_key, layer_index, qubit_indices):
        entry = gate_error_dict.get(dict_key)
        if entry is None:
            return 0.0, True 

        for qubit in qubit_indices:
            last_op[qubit] = total_time_passed

        pct = entry.get("avg_error_pct", np.nan)
        if pct is None or np.isnan(pct):
            return 0.0, False

        return pct / 100.0, False

    dag = circuit_to_dag(isa_circuit)

    layer_errors = []
    total_success_prob = 1.0
    unmatched_gates = set()

    for layer_index, layer in enumerate(dag.layers()):
        op_nodes = layer["graph"].op_nodes()
        temp_time = 0
        layer_error_rates = []
        for node in op_nodes:
            gate_name = node.op.name

            if gate_name == "barrier":
                continue

            dict_key = qiskit_name_to_dict_key.get(gate_name, gate_name)

            dur_entry = gate_durration_dict.get(dict_key)
            if dur_entry is not None and dur_entry["duration"] > temp_time:
                temp_time = dur_entry["duration"]

            qubit_indices = [dag.find_bit(q).index for q in node.qargs]
            
            err, unmatched = lookup_error(gate_name, dict_key, layer_index, qubit_indices)
            if unmatched:
                unmatched_gates.add(gate_name)

            layer_error_rates.append(err)
            total_success_prob *= (1 - err)

            # print(f"Layer {layer_index}: {gate_name} on qubits {qubit_indices}, error={err}")
        total_time_passed += temp_time
        total_success_prob *= 1-(math.e**((-1*total_time_passed)/t1))

        layer_errors.append({
            "layer": layer_index,
            "gates": [n.op.name for n in op_nodes],
            "errors": layer_error_rates,
            "layer_error_est": (
                1 - np.prod([1 - e for e in layer_error_rates])
                if layer_error_rates else 0.0
            ),
        })

    if unmatched_gates:
        print(f"[warn] unmatched gate names (treated as 0 error): {unmatched_gates}")

    return {
        "per_layer": layer_errors,
        "total_error_estimate": 1 - total_success_prob,
        "isa_circuit": isa_circuit,
        "unmatched_gates": unmatched_gates,
        "basis_gates_used": basis_gates,
    }

In [ ]:
error_dict, duration_dict = gather_stats(backend1)
print(error_dict, duration_dict)

In [ ]:
n = 10

ghz_circuit = QuantumCircuit(n)

ghz_circuit.h(0)

for qubit in range(1, n):
    ghz_circuit.cx(0, qubit)

ghz_circuit.measure_all()


In [ ]:
def esp_calc(error_dict, duration_dict):

    for n in range(2, 21):

        ghz_circuit = QuantumCircuit(n)
        ghz_circuit.h(0)

        for qubit in range(1, n):
            ghz_circuit.cx(0, qubit)

        ghz_circuit.measure_all()

        result = accumulate_circuit_error(ghz_circuit, backend1, error_dict, duration_dict)
        print(f"({n}, ({1-result["total_error_estimate"]})")

In [ ]:
esp_calc(error_dict, duration_dict)